# 🚀 nanobot 交互式教程 (Interactive Course)

欢迎来到 **一边运行，一边学习** 的交互式教程！在这个 Notebook 中，您将通过执行真实的 Python 代码，直接体验 `nanobot` 内部的核心运行流与架构机制。

请**逐个运行 (Shift + Enter)** 下面的代码块。

## 1. 初始化核心环境
首先，我们将手动导入 `nanobot.config` 中的配置，以确保环境变量被正确加载。

In [ ]:
import os
import sys
import asyncio

# 将当前目录往上回退两级，确保导入模块能找到 nanobot 核心包
sys.path.insert(0, os.path.abspath('..'))

from nanobot.config.loader import load_config, get_config_path
config = load_config(get_config_path())  # 这会加载 ~/.nanobot/config.json

print("✅ 配置文件加载成功！")
print(f"预设 Provider: {config.agents.defaults.provider}")
print(f"预设大模型版本: {config.agents.defaults.model}")

## 2. 理解 Providers 层
现在，我们要跳过命令行的封装，直接调用 `Providers` 层发起一个请求。你可以真切体会到为什么 `nanobot` 能够在不同模型间无缝切换。

In [ ]:
from nanobot.cli.commands import _make_provider

# 默认以 OpenAI 兼容协议读取你的配置 (比如 OpenRouter/DeepSeek 等)
provider = _make_provider(config)

async def test_provider():
    print("正在发送测试请求到...", config.agents.defaults.model)
    response = await provider.chat_with_retry(
        messages=[{"role": "user", "content": "你好，请用一句话介绍你自己。"}],
        model=config.agents.defaults.model,
        temperature=0.7,
    )
    print("\n🤖 [LLM 回复] ->", response.content)
    print("\n🔥 [系统消耗统计] ->", response.usage)

# 执行异步的 Provider 调用
await test_provider()

## 3. Agent Runner 核心执行流 (携带记忆与工具)
仅仅发起调用是不智能的。我们要使用 `AgentRunner` 包裹它。这是 `nanobot` 处理 **Function Calling** 自动化重入的地方。

In [ ]:
from nanobot.agent.runner import AgentRunner, AgentRunSpec
from nanobot.agent.tools.registry import ToolRegistry
from nanobot.skills.builtins.basic import get_current_time

# 注册内置工具时间查询
tools = ToolRegistry()
tools.register_callable(get_current_time)

async def test_agent():
    runner = AgentRunner(provider)
    spec = AgentRunSpec(
        initial_messages=[{"role": "user", "content": "现在几点了？"}],
        tools=tools,
        model=config.agents.defaults.model,
        max_iterations=5,
    )

    print("🌀 Agent 开始工作... (如果有输出，请观察它是如何自动读取挂载工具的)")
    result = await runner.run(spec)
    print("\n🛠️ [使用的工具集] ->", result.tools_used)
    print("\n🤖 [Agent 最终输出] ->", result.final_content)

await test_agent()

## 总结
通过上面的 3 个简单实验，你不需要深入源码，就亲自驱动了 nanobot 的 **底层 API**、以及**高层的重试与工具执行循环**。你现在可以在上面的格子里任意修改代码 (例如更换提示词、换个工具注册)，实时观摩它的表现！